In [104]:
import os
import sys
import numpy as np 
import pandas as pd
import geopandas as gpd

sys.path.append("../utils")

import config

pd.set_option("display.max_columns", None)

In [105]:
def read_geojson_with_geopandas(file_path):
        gdf = gpd.read_file(file_path)
        return gdf

insp_19 = read_geojson_with_geopandas("/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2019.geojson")
insp_20 = read_geojson_with_geopandas("/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2020.geojson")
insp_21 = read_geojson_with_geopandas("/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2021.geojson")
insp_22 = read_geojson_with_geopandas("/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2022.geojson")
insp_23 = read_geojson_with_geopandas("/capstone/wildfire_prep/data/inspections_data/cleaned_status/inspections_2023.geojson")

In [106]:
# Select only necessary columns. Columns in 2023 dataframe are different, but will be renamed in a later chunk

needed_columns = ["system_cre", "status", "latitude", "longitude", 'address_su', 'address_th', 'address__1', 'address_lo', 'address__2', 'address_ad', 'address_po', 'address_co', 'address_fu', 'Date', 'geometry']
needed_columns_23 = ["updated_at", "status", "latitude", "longitude", 'address_sub_thoroughfare', 'address_thoroughfare', 'address_suite', 'address_locality', 'address_sub_admin_area', 'address_admin_area', 'address_postal_code', 'address_country', 'address_full', 'Date', 'geometry']

In [107]:
# Narrow down all inspections dataframes to the necessary columns defined above

insp_19_clean = insp_19[needed_columns]
insp_20_clean = insp_20[needed_columns]
insp_21_clean = insp_21[needed_columns]
insp_22_clean = insp_22[needed_columns]
insp_23_clean = insp_23[needed_columns_23]

In [108]:
# Rename 2023 dataframe columns to match the others

colnames_23 = ["system_cre", "status", "latitude", "longitude", 'address_su', 'address_th', 'address__1', 'address_lo', 'address__2', 'address_ad', 'address_po', 'address_co', 'address_fu', 'Date', 'geometry']

insp_23_clean.columns = colnames_23

# verify column names change
print(insp_23_clean.columns.tolist())

['system_cre', 'status', 'latitude', 'longitude', 'address_su', 'address_th', 'address__1', 'address_lo', 'address__2', 'address_ad', 'address_po', 'address_co', 'address_fu', 'Date', 'geometry']


In [109]:
# Making copies

insp_19_clean = insp_19_clean.copy()
insp_20_clean = insp_20_clean.copy()
insp_21_clean = insp_21_clean.copy()
insp_22_clean = insp_22_clean.copy()
insp_23_clean = insp_23_clean.copy()

# Adding inspection year column

insp_19_clean['year'] = 2019
insp_20_clean['year'] = 2020
insp_21_clean['year'] = 2021
insp_22_clean['year'] = 2022
insp_23_clean['year'] = 2023

In [110]:
# Joining all inspections dataframes

all_inspections = pd.concat([insp_19_clean, insp_20_clean, insp_21_clean, insp_22_clean, insp_23_clean], axis=0, ignore_index=True)

# Fixing CRS to 3310, Albers

all_inspections = all_inspections.to_crs(epsg=3310)

print(all_inspections.crs)

# Making a copy

all_inspections_copy = all_inspections.copy()

EPSG:3310


In [111]:
# Import parcel data

parcels = os.path.join(config.data_dir, "parcel_boundaries", "cbiinputs.gdb")
parcels = gpd.read_file(parcels).to_crs(config.albers_crs)

In [112]:
# Checking parcel column names, and narrowing down the dataframe to only necessary columns

print(parcels.columns.tolist())

parcels_copy = parcels.copy()

parcel_columns = ['situs1', 'shape_leng', 'shape_Length', 'shape_Area', 'geometry']
parcels_copy = parcels[parcel_columns]

['apn', 'layer', 'situs1', 'situs2', 'acreage', 'landuse', 'usecode', 'tra', 'docnum', 'docdate', 'pcttransf', 'valreason', 'nontaxcode', 'sbeno', 'agpres', 'landvalue', 'strimpr', 'tradefix', 'livimpr', 'perpropdec', 'perspropun', 'mobilehome', 'exemptions', 'exempcode', 'homeowex', 'netsecval', 'net_impr', 'net_pers', 'net_unx', 'net_av', 'mailx1', 'mailx2', 'mnumber', 'mfrac', 'mdir', 'mstreet', 'mstrsuffix', 'munittype', 'munitnumb', 'pobox', 'mcity', 'mzip', 'mzipext', 'mstate', 'country', 'snum', 'sfra', 'sdir', 'sstreet', 'sstreetsuf', 'sunittype', 'sunitnumb', 'scity', 'szip', 'szipext', 'm_address1', 'm_address2', 'asmrollid', 'apn9', 'propid', 'sqfootage', 'yearbuilt', 'bedrooms', 'bathrooms', 'taxbill', 'permit', 'bkpglnk', 'gen_inquir', 'web_link', 'maptype', 'recmapnum', 'tractname', 'recmapbook', 'recmapdate', 'blocksecti', 'lotnum', 'unitnum', 'cbiinputs_zachary_canter_assess', 'areaunit', 'recdatetim', 'recuserid', 'recuserlog', 'section', 'township', 'range', 'shape_le

In [113]:
# Joining parcels to inspections dataframe 

inspections_parcels = gpd.sjoin(
        parcels_copy, all_inspections_copy, how="inner", predicate="contains")

# Dropping an unnecessary column, could be done more neatly 

inspections_parcels = inspections_parcels.drop(columns=['address__1'])

print(parcels_copy.crs)
print(all_inspections_copy.crs)

EPSG:3310
EPSG:3310


In [114]:
inspections_parcels

,situs1,shape_leng,shape_Length,shape_Area,geometry,index_right,system_cre,status,latitude,longitude,address_su,address_th,address_lo,address__2,address_ad,address_po,address_co,address_fu,Date,year
203,1450 CAMINO MANADERO,903.335649,903.335649,40102.354471,"MULTIPOLYGON (((16347.387 -394504.877, 16342.5...",16177,2020-05-18 16:29:44,Compliant,34.466073,-119.821791,1450,Camino Manadero,Santa Barbara,Santa Barbara,CA,93111,US,1450 Camino Manadero Santa Barbara Santa Barba...,2020-06-27,2020
203,1450 CAMINO MANADERO,903.335649,903.335649,40102.354471,"MULTIPOLYGON (((16347.387 -394504.877, 16342.5...",1472,2019-05-28 15:21:11,Compliant,34.466073,-119.821791,1450,Camino Manadero,Santa Barbara,Santa Barbara,CA,93111,US,1450 Camino Manadero Santa Barbara Santa Barba...,2019-06-25,2019
203,1450 CAMINO MANADERO,903.335649,903.335649,40102.354471,"MULTIPOLYGON (((16347.387 -394504.877, 16342.5...",28016,2021-04-05 08:56:31,Compliant,34.466073,-119.821791,1450,Camino Manadero,Santa Barbara,Santa Barbara,CA,93111,US,1450 Camino Manadero Santa Barbara Santa Barba...,2021-06-26,2021
203,1450 CAMINO MANADERO,903.335649,903.335649,40102.354471,"MULTIPOLYGON (((16347.387 -394504.877, 16342.5...",46334,2022-04-06 09:59:33,Compliant,34.466073,-119.821791,1450,Camino Manadero,Santa Barbara,Santa Barbara,CA,93111,US,1450 Camino Manadero Santa Barbara Santa Barba...,2022-07-05,2022
205,1475 CAMINO MELENO,1113.961402,1113.961402,57984.774296,"MULTIPOLYGON (((16811.85 -394379.557, 16800.14...",1538,2019-05-28 15:21:27,Compliant,34.466445,-119.817230,1475,Camino Meleno,Santa Barbara,Santa Barbara,CA,93111,US,1475 Camino Meleno Santa Barbara Santa Barbara...,2019-06-25,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132303,2685 HWY 33,2962.102000,2962.102000,482989.154940,"MULTIPOLYGON (((46691.776 -350305.727, 46682.3...",54407,2023-08-22 17:28:10,Compliant,34.864993,-119.489451,2685,Highway 33,Maricopa,Santa Barbara,CA,93252,US,2685 Highway 33 Maricopa Santa Barbara CA 9325...,2023-08-22,2023
132303,2685 HWY 33,2962.102000,2962.102000,482989.154940,"MULTIPOLYGON (((46691.776 -350305.727, 46682.3...",40773,2022-04-05 13:47:50,Compliant,34.864993,-119.489451,2685,Highway 33,Maricopa,Santa Barbara,CA,93252,US,2685 Highway 33 Maricopa Santa Barbara CA 9325...,2022-06-28,2022
132303,2685 HWY 33,2962.102000,2962.102000,482989.154940,"MULTIPOLYGON (((46691.776 -350305.727, 46682.3...",36281,2021-04-05 09:30:25,Compliant,34.864993,-119.489451,2685,Highway 33,Maricopa,Santa Barbara,CA,93252,US,2685 Highway 33 Maricopa Santa Barbara CA 9325...,2021-08-17,2021
132303,2685 HWY 33,2962.102000,2962.102000,482989.154940,"MULTIPOLYGON (((46691.776 -350305.727, 46682.3...",9958,2019-05-28 16:13:31,Compliant,34.864993,-119.489451,2685,Highway 33,Maricopa,Santa Barbara,CA,93252,US,2685 Highway 33 Maricopa Santa Barbara CA 9325...,2019-06-11,2019


In [115]:
buffered_buildings = read_geojson_with_geopandas('/capstone/wildfire_prep/data/joined_inspections_parcels/buffered_buildings_2022.geojson')

In [116]:
buffered_buildings_copy = buffered_buildings.copy()

print(buffered_buildings.columns.tolist())

buff_buildings_columns = ['geometry']

buffered_buildings_copy = buffered_buildings_copy[buff_buildings_columns]

['release', 'capture_dates_range', 'STATEFP', 'COUNTYFP', 'COUNTYNS', 'GEOID', 'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'Shape_Leng', 'Shape_Area', 'index_right', 'apn_left_1', 'layer_1', 'situs1_1', 'situs2_1', 'acreage_1', 'landuse_1', 'usecode_1', 'tra_1', 'docnum_1', 'docdate_1', 'pcttransf_1', 'valreason_1', 'nontaxcode_1', 'sbeno_1', 'agpres_1', 'landvalue_1', 'strimpr_1', 'tradefix_1', 'livimpr_1', 'perpropdec_1', 'perspropun_1', 'mobilehome_1', 'exemptions_1', 'exempcode_1', 'homeowex_1', 'netsecval_1', 'net_impr_1', 'net_pers_1', 'net_unx_1', 'net_av_1', 'mailx1_1', 'mailx2_1', 'mnumber_1', 'mfrac_1', 'mdir_1', 'mstreet_1', 'mstrsuffix_1', 'munittype_1', 'munitnumb_1', 'pobox_1', 'mcity_1', 'mzip_1', 'mzipext_1', 'mstate_1', 'country_1', 'snum_1', 'sfra_1', 'sdir_1', 'sstreet_1', 'sstreetsuf_1', 'sunittype_1', 'sunitnumb_1', 'scity_1', 'szip_1', 'szipext_1', 'm_address1_1', 'm_address2

In [117]:
buffered_buildings_copy

,geometry
0,"POLYGON ((-22767.476 -375448.119, -23014.787 -..."
1,"POLYGON ((-9290.574 -378632.312, -9035.561 -37..."
2,"POLYGON ((-38766.251 -371421.02, -38538.65 -37..."
3,"POLYGON ((-11561.332 -372335.927, -11718.559 -..."
4,"POLYGON ((-23742.879 -348820.427, -23927.393 -..."
...,...
9030,"POLYGON ((-42570.319 -369065.356, -42570.319 -..."
9031,"POLYGON ((-42539.328 -367092.847, -42539.446 -..."
9032,"POLYGON ((-8759.709 -378740.536, -8759.709 -37..."
9033,"POLYGON ((-42551.126 -367015.662, -42551.159 -..."


In [118]:
inspections_buffbuildings = gpd.sjoin(
        buffered_buildings_copy, all_inspections_copy, how="inner", predicate="contains")

In [119]:
inspections_buffbuildings

,geometry,index_right,system_cre,status,latitude,longitude,address_su,address_th,address__1,address_lo,address__2,address_ad,address_po,address_co,address_fu,Date,year
1,"POLYGON ((-9290.574 -378632.312, -9035.561 -37...",37274,2021-04-05 09:33:18,Compliant,34.608120,-120.099620,2975,Mission Dr,None,Solvang,Santa Barbara,CA,93463,US,2975 Mission Dr Solvang Santa Barbara CA 93463 US,2021-07-08,2021
1,"POLYGON ((-9290.574 -378632.312, -9035.561 -37...",10929,2019-05-28 16:18:04,Compliant,34.608120,-120.099620,2975,Mission Dr,None,Solvang,Santa Barbara,CA,93463,US,2975 Mission Dr Solvang Santa Barbara CA 93463 US,2019-10-10,2019
1,"POLYGON ((-9290.574 -378632.312, -9035.561 -37...",23982,2020-05-19 15:01:53,Compliant,34.608120,-120.099620,2975,Mission Dr,None,Solvang,Santa Barbara,CA,93463,US,2975 Mission Dr Solvang Santa Barbara CA 93463 US,2020-07-10,2020
1,"POLYGON ((-9290.574 -378632.312, -9035.561 -37...",41709,2022-04-05 13:50:56,Compliant,34.608120,-120.099620,2975,Mission Dr,None,Solvang,Santa Barbara,CA,93463,US,2975 Mission Dr Solvang Santa Barbara CA 93463 US,2022-08-15,2022
1,"POLYGON ((-9290.574 -378632.312, -9035.561 -37...",55317,2023-08-15 12:41:48,Compliant,34.608120,-120.099620,2975,Mission Dr,None,Solvang,Santa Barbara,CA,93463,US,2975 Mission Dr Solvang Santa Barbara CA 93463 US,2023-08-14,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8945,"POLYGON ((22007.218 -395742.615, 22004.059 -39...",45920,2022-04-06 09:58:21,Compliant,34.454282,-119.760424,4120,Via Andorra,None,Santa Barbara,Santa Barbara,CA,93110,US,4120 Via Andorra Santa Barbara Santa Barbara C...,2022-06-29,2022
8945,"POLYGON ((22007.218 -395742.615, 22004.059 -39...",59408,2023-06-21 17:00:00,Compliant,34.454282,-119.760424,4120,Via Andorra,None,Santa Barbara,Santa Barbara,CA,93110,US,4120 Via Andorra Santa Barbara Santa Barbara C...,2023-06-21,2023
8945,"POLYGON ((22007.218 -395742.615, 22004.059 -39...",27596,2021-04-05 08:55:38,Compliant,34.454282,-119.760424,4120,Via Andorra,None,Santa Barbara,Santa Barbara,CA,93110,US,4120 Via Andorra Santa Barbara Santa Barbara C...,2021-06-12,2021
8945,"POLYGON ((22007.218 -395742.615, 22004.059 -39...",1053,2019-05-28 15:19:48,Compliant,34.454282,-119.760424,4120,Via Andorra,None,Santa Barbara,Santa Barbara,CA,93110,US,4120 Via Andorra Santa Barbara Santa Barbara C...,2019-06-07,2019


In [120]:
inspections_parcels

,situs1,shape_leng,shape_Length,shape_Area,geometry,index_right,system_cre,status,latitude,longitude,address_su,address_th,address_lo,address__2,address_ad,address_po,address_co,address_fu,Date,year
203,1450 CAMINO MANADERO,903.335649,903.335649,40102.354471,"MULTIPOLYGON (((16347.387 -394504.877, 16342.5...",16177,2020-05-18 16:29:44,Compliant,34.466073,-119.821791,1450,Camino Manadero,Santa Barbara,Santa Barbara,CA,93111,US,1450 Camino Manadero Santa Barbara Santa Barba...,2020-06-27,2020
203,1450 CAMINO MANADERO,903.335649,903.335649,40102.354471,"MULTIPOLYGON (((16347.387 -394504.877, 16342.5...",1472,2019-05-28 15:21:11,Compliant,34.466073,-119.821791,1450,Camino Manadero,Santa Barbara,Santa Barbara,CA,93111,US,1450 Camino Manadero Santa Barbara Santa Barba...,2019-06-25,2019
203,1450 CAMINO MANADERO,903.335649,903.335649,40102.354471,"MULTIPOLYGON (((16347.387 -394504.877, 16342.5...",28016,2021-04-05 08:56:31,Compliant,34.466073,-119.821791,1450,Camino Manadero,Santa Barbara,Santa Barbara,CA,93111,US,1450 Camino Manadero Santa Barbara Santa Barba...,2021-06-26,2021
203,1450 CAMINO MANADERO,903.335649,903.335649,40102.354471,"MULTIPOLYGON (((16347.387 -394504.877, 16342.5...",46334,2022-04-06 09:59:33,Compliant,34.466073,-119.821791,1450,Camino Manadero,Santa Barbara,Santa Barbara,CA,93111,US,1450 Camino Manadero Santa Barbara Santa Barba...,2022-07-05,2022
205,1475 CAMINO MELENO,1113.961402,1113.961402,57984.774296,"MULTIPOLYGON (((16811.85 -394379.557, 16800.14...",1538,2019-05-28 15:21:27,Compliant,34.466445,-119.817230,1475,Camino Meleno,Santa Barbara,Santa Barbara,CA,93111,US,1475 Camino Meleno Santa Barbara Santa Barbara...,2019-06-25,2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132303,2685 HWY 33,2962.102000,2962.102000,482989.154940,"MULTIPOLYGON (((46691.776 -350305.727, 46682.3...",54407,2023-08-22 17:28:10,Compliant,34.864993,-119.489451,2685,Highway 33,Maricopa,Santa Barbara,CA,93252,US,2685 Highway 33 Maricopa Santa Barbara CA 9325...,2023-08-22,2023
132303,2685 HWY 33,2962.102000,2962.102000,482989.154940,"MULTIPOLYGON (((46691.776 -350305.727, 46682.3...",40773,2022-04-05 13:47:50,Compliant,34.864993,-119.489451,2685,Highway 33,Maricopa,Santa Barbara,CA,93252,US,2685 Highway 33 Maricopa Santa Barbara CA 9325...,2022-06-28,2022
132303,2685 HWY 33,2962.102000,2962.102000,482989.154940,"MULTIPOLYGON (((46691.776 -350305.727, 46682.3...",36281,2021-04-05 09:30:25,Compliant,34.864993,-119.489451,2685,Highway 33,Maricopa,Santa Barbara,CA,93252,US,2685 Highway 33 Maricopa Santa Barbara CA 9325...,2021-08-17,2021
132303,2685 HWY 33,2962.102000,2962.102000,482989.154940,"MULTIPOLYGON (((46691.776 -350305.727, 46682.3...",9958,2019-05-28 16:13:31,Compliant,34.864993,-119.489451,2685,Highway 33,Maricopa,Santa Barbara,CA,93252,US,2685 Highway 33 Maricopa Santa Barbara CA 9325...,2019-06-11,2019


In [121]:
print("Percent of values in columns that are unique: \n")
for col in insp_21:
    print(
        f"{col}: {
            round(
                (insp_21[col].unique().shape[0])/(len(insp_21)) * 100, 
                4)
            }%"
    )

Percent of values in columns that are unique: 

fulcrum_id: 100.0%
created_at: 28.0403%
updated_at: 99.7788%
system_cre: 14.9657%
system_upd: 98.694%
version: 0.0571%
status: 0.0143%
project: 0.0071%
assigned_t: 0.0071%
latitude: 96.8598%
longitude: 97.2524%
report_tit: 0.0071%
globalid: 0.0071%
keyid: 0.0071%
propertyst: 0.0071%
inspection: 0.0071%
inspectorp: 0.0214%
inspecti_1: 0.8279%
addressvis: 0.0285%
address_su: 35.2269%
address_th: 7.3223%
address__1: 0.0357%
address_lo: 0.157%
address__2: 0.0214%
address_ad: 0.0357%
address_po: 0.1213%
address_co: 0.0357%
address_fu: 86.8113%
calfireuni: 0.0071%
county: 0.0071%
community: 0.4353%
community_: 0.0071%
battalion: 0.0214%
enginenumb: 0.1713%
stationnam: 0.1142%
shift: 0.0428%
accessegre: 0.0143%
occupantho: 0.0143%
deliveryno: 0.0214%
inspecti_2: 0.0357%
a_removebr: 0.0143%
b_removele: 0.0143%
c_removede: 0.0143%
d_removede: 0.0143%
e_removefl: 0.0143%
f_removefl: 0.0143%
g_relocate: 0.0143%
h_cutannua: 0.0143%
i_removefu: 0.0143

#### Josh's opinion

Keep columns by default if they have (or at least mark them for likely usefulness)...
* % unique values > 0.1% 
* % unique values < 80%

For the rest of the columns, investigate whether to keep them on a case by case basis...
* high uniqueness may indicate some kind of unique identifier
* low uniqueness column may still be important